# 03 — Batch-extract UCF101 poses and build LSTM sequences

This notebook covers the next two stages:

1. **Batch pose extraction:** automatically process a balanced subset of five UCF101 classes and save one `.npz` file per source video.
2. **Sequence dataset creation:** build fixed-length pose windows and save train/validation/test arrays for the LSTM.

## Important settings
- Start with **10 videos per class** (50 videos total). The notebook can be rerun; completed videos are skipped and partial videos resume from saved sampled frames.
- The current stable baseline uses **CPU inference**, samples every 4th original frame, runs YOLO person detection with a fallback threshold, and runs the already-trained ViTPose checkpoint. It does not train ViTPose.
- To avoid the earlier kernel crashes, this batch version **does not run ByteTrack or write annotated MP4 videos**. It selects the largest detected person independently in each sampled frame. This is a baseline, not guaranteed persistent identity tracking. We'll add tracking after batch extraction is stable.
- Raw videos are never modified.

## Run order
Run cells from top to bottom. Cell 5 is the batch loop; run it once and let it finish. If the kernel restarts, restart the kernel, rerun Cells 1–4, then rerun Cell 5. Completed `.npz` files are skipped; partial files resume.

After extraction finishes, run the remaining cells to build sequences. The LSTM itself is trained later in `04_train_lstm.ipynb`.

In [1]:
from pathlib import Path
import sys
import re
import gc
import time
import hashlib
import json
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
sys.path.insert(0, str(PROJECT_DIR))

from src.vitpose_model import (
    ViTPose,
    get_affine_transform,
    decode_heatmaps,
    draw_skeleton_on,
    IMG_H,
    IMG_W,
    IMG_MEAN,
    IMG_STD,
)

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "epoch_209.pth"
YOLO_PATH = PROJECT_DIR / "notebooks" / "yolo11n.pt"
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"

assert CHECKPOINT_PATH.is_file(), f"ViTPose checkpoint not found: {CHECKPOINT_PATH}"
assert YOLO_PATH.is_file(), f"YOLO model file not found: {YOLO_PATH}"
assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"

# Keep CPU inference for this stable/resumable batch pilot.
DEVICE = "cpu"
torch.set_num_threads(2)

print("Loading ViTPose checkpoint on CPU...", flush=True)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = ViTPose()
model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()
SAVED_EPOCH = checkpoint.get("epoch", "unknown")
del checkpoint
gc.collect()
print("ViTPose loaded. Saved epoch:", SAVED_EPOCH, flush=True)

print("Loading YOLO detector...", flush=True)
detector = YOLO(str(YOLO_PATH))
print("YOLO loaded on CPU.", flush=True)

Loading ViTPose checkpoint on CPU...
ViTPose loaded. Saved epoch: 209
Loading YOLO detector...
YOLO loaded on CPU.


In [2]:
# ---------- User-adjustable batch settings ----------
SELECTED_CLASSES = [
    "Biking",
    "BoxingPunchingBag",
    "TennisSwing",
    "BenchPress",
    "HorseRiding",
]

N_VIDEOS_PER_CLASS = 10   # 5 classes x 10 videos = at most 50 videos
RANDOM_SEED = 42
FRAME_STEP = 4            # run YOLO + ViTPose every 4th original frame
DETECTION_CONF = 0.25
DETECTION_CONF_FALLBACK = 0.10  # retry only if primary confidence finds no person
YOLO_IMGSZ = 640
POSE_DRAW_THRESHOLD = 0.10
SAVE_PARTIAL_EVERY_SAMPLES = 5
SAVE_ONE_PREVIEW_PER_VIDEO = True

DATA_DIR = PROJECT_DIR / "data" / "raw" / "ucf_sports"  # folder contains UCF101 classes
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
BATCH_KEYPOINT_DIR = PROCESSED_DIR / "keypoints_batch_5class"
PREVIEW_DIR = PROJECT_DIR / "outputs" / "pose_videos" / "batch_5class_previews"
SELECTED_MANIFEST_PATH = PROCESSED_DIR / "selected_5class_videos.csv"
EXTRACTION_REPORT_PATH = PROCESSED_DIR / "batch_5class_extraction_report.csv"
SEQUENCE_DIR = PROCESSED_DIR / "sequences_5class"

for folder in [BATCH_KEYPOINT_DIR, PREVIEW_DIR, PROCESSED_DIR, SEQUENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)
required_cols = {"video_path", "label"}
missing_cols = required_cols - set(manifest.columns)
assert not missing_cols, f"Manifest is missing columns: {missing_cols}"
manifest["label"] = manifest["label"].astype(str)

print("Project:", PROJECT_DIR)
print("Manifest videos:", len(manifest))
print("Classes to process:", SELECTED_CLASSES)
print("Videos per class:", N_VIDEOS_PER_CLASS)
print("Maximum selected videos:", len(SELECTED_CLASSES) * N_VIDEOS_PER_CLASS)
print("Keypoint output:", BATCH_KEYPOINT_DIR)
print("Sequence output:", SEQUENCE_DIR)

Project: C:\Users\admin\ViTPose_Action_Recognition
Manifest videos: 13320
Classes to process: ['Biking', 'BoxingPunchingBag', 'TennisSwing', 'BenchPress', 'HorseRiding']
Videos per class: 10
Maximum selected videos: 50
Keypoint output: C:\Users\admin\ViTPose_Action_Recognition\data\processed\keypoints_batch_5class
Sequence output: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_5class


In [3]:
def recording_group(video_path):
    """Extract UCF-style recording group gNN to diversify the selected videos."""
    stem = Path(str(video_path)).stem
    match = re.search(r"_g(\d+)_", stem, flags=re.IGNORECASE)
    return f"g{match.group(1)}" if match else stem


def choose_class_videos(class_df, n, seed):
    """Choose one video per recording group first; fill from remaining videos if needed."""
    class_df = class_df.copy().sort_values("video_path").reset_index(drop=True)
    class_df["recording_group"] = class_df["video_path"].map(recording_group)
    representatives = class_df.drop_duplicates("recording_group", keep="first")
    if len(representatives) >= n:
        return representatives.sample(n=n, random_state=seed).sort_values("video_path")
    chosen = representatives.copy()
    needed = n - len(chosen)
    remaining = class_df[~class_df["video_path"].isin(chosen["video_path"])]
    if needed > 0 and len(remaining) > 0:
        chosen = pd.concat([chosen, remaining.sample(n=min(needed, len(remaining)), random_state=seed)])
    return chosen.sort_values("video_path").reset_index(drop=True)

selected_parts = []
for class_offset, class_name in enumerate(SELECTED_CLASSES):
    class_df = manifest[manifest["label"].str.casefold().eq(class_name.casefold())].copy()
    if class_df.empty:
        raise ValueError(
            f"Class {class_name!r} was not found in manifest. "
            f"Available examples: {sorted(manifest['label'].unique())[:20]}"
        )
    class_selected = choose_class_videos(class_df, N_VIDEOS_PER_CLASS, RANDOM_SEED + class_offset)
    class_selected["label"] = class_name  # consistent display label
    selected_parts.append(class_selected)

selected_videos = pd.concat(selected_parts, ignore_index=True)
selected_videos["video_exists"] = selected_videos["video_path"].map(lambda p: Path(str(p)).is_file())

print("Selected videos per class:")
display(selected_videos.groupby("label").size().rename("selected_videos").reset_index())
print("Missing video files:", int((~selected_videos["video_exists"]).sum()))
if (~selected_videos["video_exists"]).any():
    display(selected_videos.loc[~selected_videos["video_exists"], ["label", "video_path"]])

selected_videos = selected_videos[selected_videos["video_exists"]].reset_index(drop=True)
selected_videos.to_csv(SELECTED_MANIFEST_PATH, index=False)
print("Saved selection manifest:", SELECTED_MANIFEST_PATH)
print("Videos to process:", len(selected_videos))

Selected videos per class:


,label,selected_videos
0,BenchPress,10
1,Biking,10
2,BoxingPunchingBag,10
3,HorseRiding,10
4,TennisSwing,10


Missing video files: 0
Saved selection manifest: C:\Users\admin\ViTPose_Action_Recognition\data\processed\selected_5class_videos.csv
Videos to process: 50


In [4]:
def pose_for_box_cpu(frame_rgb, bbox):
    """Estimate COCO-17 keypoints for one person's crop using the original model helpers."""
    x1, y1, x2, y2 = map(float, bbox)
    h, w = frame_rgb.shape[:2]
    x1 = float(np.clip(x1, 0, w - 1))
    x2 = float(np.clip(x2, 0, w))
    y1 = float(np.clip(y1, 0, h - 1))
    y2 = float(np.clip(y2, 0, h))
    bw, bh = x2 - x1, y2 - y1
    if bw < 2 or bh < 2:
        return None

    center = np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=np.float32)
    scale = max(bw, bh * IMG_W / IMG_H) * 1.25
    transform = get_affine_transform(center, scale, 0.0, (IMG_W, IMG_H))
    crop = cv2.warpAffine(frame_rgb, transform, (IMG_W, IMG_H))
    crop = crop.astype(np.float32) / 255.0
    crop = (crop - IMG_MEAN) / IMG_STD
    tensor = torch.from_numpy(crop.transpose(2, 0, 1).copy()).float().unsqueeze(0)

    with torch.inference_mode():
        heatmaps = model(tensor)[0].float().cpu().numpy()
    keypoints = decode_heatmaps(heatmaps, center, scale).astype(np.float32)

    del tensor, heatmaps, crop
    if keypoints.shape != (17, 3):
        return None
    return keypoints


def safe_output_stem(label, video_path):
    safe_label = re.sub(r"[^A-Za-z0-9_-]+", "_", str(label)).strip("_") or "unknown"
    vp = Path(str(video_path))
    digest = hashlib.sha1(str(vp.resolve()).encode("utf-8")).hexdigest()[:10]
    return f"{safe_label}__{vp.stem}__{digest}"


def save_observations(npz_path, observations, label, video_path, fps, total_frames, frame_step):
    """Atomically save the currently completed sampled-frame observations."""
    if not observations:
        return
    arrays = {
        "keypoints": np.stack([o["keypoints"] for o in observations]).astype(np.float32),
        "boxes": np.stack([o["bbox"] for o in observations]).astype(np.float32),
        "frame_indices": np.asarray([o["frame_index"] for o in observations], dtype=np.int32),
        "detection_found": np.asarray([o["detection_found"] for o in observations], dtype=np.bool_),
        "has_person": np.asarray([o["pose_ok"] for o in observations], dtype=np.bool_),
        "confidence_used": np.asarray([o["confidence_used"] for o in observations], dtype=np.float32),
        "fallback_used": np.asarray([o["fallback_used"] for o in observations], dtype=np.bool_),
        "label": np.asarray(str(label)),
        "video_path": np.asarray(str(video_path)),
        "fps": np.asarray(float(fps), dtype=np.float32),
        "total_frames": np.asarray(int(total_frames), dtype=np.int32),
        "frame_step": np.asarray(int(frame_step), dtype=np.int32),
        "person_selection": np.asarray("largest_bbox_each_sampled_frame_no_tracking"),
    }
    tmp_path = npz_path.with_name(npz_path.stem + ".tmp.npz")
    np.savez_compressed(tmp_path, **arrays)
    tmp_path.replace(npz_path)


def load_observations(npz_path):
    """Load completed records from a partial NPZ to resume after interruption."""
    if not npz_path.is_file():
        return []
    with np.load(npz_path, allow_pickle=False) as d:
        keys = d["keypoints"].copy()
        boxes = d["boxes"].copy()
        indices = d["frame_indices"].copy()
        detected = d["detection_found"].copy()
        pose_ok = d["has_person"].copy()
        conf_used = d["confidence_used"].copy()
        fallback = d["fallback_used"].copy()
    return [
        {
            "keypoints": keys[i],
            "bbox": boxes[i],
            "frame_index": int(indices[i]),
            "detection_found": bool(detected[i]),
            "pose_ok": bool(pose_ok[i]),
            "confidence_used": float(conf_used[i]),
            "fallback_used": bool(fallback[i]),
        }
        for i in range(len(indices))
    ]


def build_report_row(video_path, label, status, observations, elapsed, output_path, error=""):
    total = len(observations)
    detections = sum(o["detection_found"] for o in observations)
    poses = sum(o["pose_ok"] for o in observations)
    fallback = sum(o["fallback_used"] for o in observations)
    return {
        "label": str(label),
        "video_path": str(video_path),
        "status": status,
        "sampled_frames": total,
        "detections_found": detections,
        "pose_successes": poses,
        "fallback_used_frames": fallback,
        "detection_rate": detections / total if total else 0.0,
        "pose_success_rate": poses / total if total else 0.0,
        "elapsed_seconds": round(float(elapsed), 1),
        "output_npz": str(output_path) if output_path else "",
        "error": error,
    }

print("Extraction helper functions defined.")

Extraction helper functions defined.


## Stage A — Batch-extract keypoints from 50 videos

The following cell processes the selected list automatically. You do **not** need to rerun it for every individual video. It writes a per-video report after each video and partial observations every few sampled frames.

**Identity limitation:** this version selects the largest detected person independently in each sampled frame. It does not use ByteTrack, so a clip containing multiple people could switch between people. We are using this baseline because the tracking version caused kernel crashes; tracking will be reintroduced after extraction is stable.

In [5]:
report_rows = []
run_started = time.time()

for video_position, (_, row) in enumerate(selected_videos.iterrows(), start=1):
    video_path = Path(str(row["video_path"]))
    label = str(row["label"])
    out_stem = safe_output_stem(label, video_path)
    final_npz = BATCH_KEYPOINT_DIR / f"{out_stem}.npz"
    partial_npz = BATCH_KEYPOINT_DIR / f"{out_stem}.partial.npz"
    preview_path = PREVIEW_DIR / f"{out_stem}_preview.jpg"

    print(f"\n[{video_position}/{len(selected_videos)}] {label}: {video_path.name}", flush=True)

    # Skip already-completed output files.
    if final_npz.is_file():
        try:
            loaded = load_observations(final_npz)
            report_row = build_report_row(video_path, label, "skipped_existing", loaded, 0, final_npz)
            print("  Existing complete NPZ found; skipping.", flush=True)
        except Exception as exc:
            print("  Existing NPZ could not be read; re-extracting:", repr(exc), flush=True)
            final_npz.unlink(missing_ok=True)
            report_row = None
        if report_row is not None:
            report_rows.append(report_row)
            pd.DataFrame(report_rows).to_csv(EXTRACTION_REPORT_PATH, index=False)
            continue

    observations = load_observations(partial_npz)
    already_done = {o["frame_index"] for o in observations}
    print("  Saved sampled frames to resume from:", len(observations), flush=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        report_row = build_report_row(video_path, label, "open_failed", observations, 0, partial_npz, "OpenCV could not open video")
        report_rows.append(report_row)
        pd.DataFrame(report_rows).to_csv(EXTRACTION_REPORT_PATH, index=False)
        cap.release()
        continue

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    total_frames_reported = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    frame_index = 0
    new_samples_since_save = 0
    started = time.time()
    first_preview_saved = preview_path.is_file()
    video_error = ""

    try:
        while True:
            ok, frame_bgr = cap.read()
            if not ok:
                break

            if frame_index % FRAME_STEP == 0 and frame_index not in already_done:
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                det = detector.predict(
                    frame_bgr,
                    classes=[0],
                    conf=DETECTION_CONF,
                    imgsz=YOLO_IMGSZ,
                    device=DEVICE,
                    verbose=False,
                )[0]
                confidence_used = DETECTION_CONF
                fallback_used = False

                if det.boxes is None or len(det.boxes) == 0:
                    det = detector.predict(
                        frame_bgr,
                        classes=[0],
                        conf=DETECTION_CONF_FALLBACK,
                        imgsz=YOLO_IMGSZ,
                        device=DEVICE,
                        verbose=False,
                    )[0]
                    confidence_used = DETECTION_CONF_FALLBACK
                    fallback_used = True

                detection_found = det.boxes is not None and len(det.boxes) > 0
                bbox = np.full((4,), np.nan, dtype=np.float32)
                keypoints = np.full((17, 3), np.nan, dtype=np.float32)
                pose_ok = False

                if detection_found:
                    boxes = det.boxes.xyxy.cpu().numpy()
                    areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
                    bbox = boxes[int(np.argmax(areas))].astype(np.float32)
                    try:
                        estimated = pose_for_box_cpu(frame_rgb, bbox)
                        if estimated is not None and estimated.shape == (17, 3):
                            keypoints = estimated
                            pose_ok = bool(np.isfinite(keypoints[:, :2]).all(axis=1).any())
                    except Exception as exc:
                        print(f"  Pose error at frame {frame_index}: {exc!r}", flush=True)

                    if pose_ok and SAVE_ONE_PREVIEW_PER_VIDEO and not first_preview_saved:
                        preview_rgb = draw_skeleton_on(frame_rgb.copy(), keypoints, thresh=POSE_DRAW_THRESHOLD)
                        x1, y1, x2, y2 = map(int, bbox)
                        cv2.rectangle(preview_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        preview_path.parent.mkdir(parents=True, exist_ok=True)
                        cv2.imwrite(str(preview_path), cv2.cvtColor(preview_rgb, cv2.COLOR_RGB2BGR))
                        first_preview_saved = True

                observations.append({
                    "keypoints": keypoints.astype(np.float32),
                    "bbox": bbox.astype(np.float32),
                    "frame_index": int(frame_index),
                    "detection_found": bool(detection_found),
                    "pose_ok": bool(pose_ok),
                    "confidence_used": float(confidence_used),
                    "fallback_used": bool(fallback_used),
                })
                already_done.add(frame_index)
                new_samples_since_save += 1

                if len(observations) % 10 == 0:
                    print(
                        f"  samples={len(observations)} | detections={sum(o['detection_found'] for o in observations)} "
                        f"| pose successes={sum(o['pose_ok'] for o in observations)}",
                        flush=True,
                    )

                if new_samples_since_save >= SAVE_PARTIAL_EVERY_SAMPLES:
                    save_observations(partial_npz, observations, label, video_path, fps, total_frames_reported, FRAME_STEP)
                    new_samples_since_save = 0

                del frame_rgb, det
                if len(observations) % 10 == 0:
                    gc.collect()

            frame_index += 1

        # All original frames were read to EOF: extraction for this video is complete.
        save_observations(partial_npz, observations, label, video_path, fps, total_frames_reported, FRAME_STEP)
        if not observations:
            report_row = build_report_row(video_path, label, "no_samples", observations, time.time() - started, partial_npz, "No sampled frames extracted")
        else:
            partial_npz.replace(final_npz)
            report_row = build_report_row(video_path, label, "success", observations, time.time() - started, final_npz)

    except Exception as exc:
        video_error = repr(exc)
        # Preserve whatever was gathered before a normal Python exception.
        if observations:
            save_observations(partial_npz, observations, label, video_path, fps, total_frames_reported, FRAME_STEP)
        report_row = build_report_row(video_path, label, "error_partial_saved", observations, time.time() - started, partial_npz, video_error)
        print("  ERROR; partial file retained:", video_error, flush=True)

    finally:
        cap.release()
        gc.collect()

    report_rows.append(report_row)
    pd.DataFrame(report_rows).to_csv(EXTRACTION_REPORT_PATH, index=False)
    print(
        f"  Status={report_row['status']} | samples={report_row['sampled_frames']} | "
        f"poses={report_row['pose_successes']} | elapsed={report_row['elapsed_seconds']}s",
        flush=True,
    )

print("\nBATCH EXTRACTION CELL FINISHED", flush=True)
print("Total elapsed minutes:", round((time.time() - run_started) / 60, 1), flush=True)
print("Report:", EXTRACTION_REPORT_PATH, flush=True)
display(pd.DataFrame(report_rows))


[1/50] Biking: v_Biking_g01_c01.avi
  Saved sampled frames to resume from: 0
  samples=10 | detections=10 | pose successes=10
  samples=20 | detections=20 | pose successes=20
  samples=30 | detections=30 | pose successes=30
  Status=success | samples=38 | poses=38 | elapsed=7.6s

[2/50] Biking: v_Biking_g02_c01.avi
  Saved sampled frames to resume from: 0
  samples=10 | detections=3 | pose successes=3
  samples=20 | detections=3 | pose successes=3
  samples=30 | detections=8 | pose successes=8
  samples=40 | detections=14 | pose successes=14
  samples=50 | detections=19 | pose successes=19
  samples=60 | detections=26 | pose successes=26
  Status=success | samples=60 | poses=26 | elapsed=5.9s

[3/50] Biking: v_Biking_g06_c01.avi
  Saved sampled frames to resume from: 0
  samples=10 | detections=6 | pose successes=6
  samples=20 | detections=15 | pose successes=15
  Status=success | samples=26 | poses=17 | elapsed=2.8s

[4/50] Biking: v_Biking_g09_c01.avi
  Saved sampled frames to resu

,label,video_path,status,sampled_frames,detections_found,pose_successes,fallback_used_frames,detection_rate,pose_success_rate,elapsed_seconds,output_npz,error
0,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,38,38,38,15,1.000000,1.000000,7.6,C:\Users\admin\ViTPose_Action_Recognition\data...,
1,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,60,26,26,45,0.433333,0.433333,5.9,C:\Users\admin\ViTPose_Action_Recognition\data...,
2,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,26,17,17,14,0.653846,0.653846,2.8,C:\Users\admin\ViTPose_Action_Recognition\data...,
3,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,45,40,40,11,0.888889,0.888889,5.2,C:\Users\admin\ViTPose_Action_Recognition\data...,
4,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,60,56,56,8,0.933333,0.933333,7.2,C:\Users\admin\ViTPose_Action_Recognition\data...,
5,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,60,27,27,50,0.450000,0.450000,6.1,C:\Users\admin\ViTPose_Action_Recognition\data...,
6,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,32,32,32,0,1.000000,1.000000,3.8,C:\Users\admin\ViTPose_Action_Recognition\data...,
7,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,38,36,36,7,0.947368,0.947368,4.5,C:\Users\admin\ViTPose_Action_Recognition\data...,
8,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,134,130,130,11,0.970149,0.970149,15.7,C:\Users\admin\ViTPose_Action_Recognition\data...,
9,Biking,C:\Users\admin\ViTPose_Action_Recognition\data...,success,45,41,41,11,0.911111,0.911111,5.3,C:\Users\admin\ViTPose_Action_Recognition\data...,


In [6]:
# Extraction audit: complete NPZ files and report status
report = pd.read_csv(EXTRACTION_REPORT_PATH) if EXTRACTION_REPORT_PATH.is_file() else pd.DataFrame()
complete_files = [p for p in BATCH_KEYPOINT_DIR.glob("*.npz") if ".partial." not in p.name and ".tmp." not in p.name]
partial_files = list(BATCH_KEYPOINT_DIR.glob("*.partial.npz"))

print("Completed NPZ files:", len(complete_files))
print("Partial NPZ files:", len(partial_files))
if not report.empty:
    print("\nStatus counts:")
    display(report["status"].value_counts().rename_axis("status").reset_index(name="videos"))
    print("\nPer-class extraction summary:")
    display(report.groupby("label").agg(
        videos=("video_path", "count"),
        pose_successes=("pose_successes", "sum"),
        sampled_frames=("sampled_frames", "sum"),
        mean_pose_success_rate=("pose_success_rate", "mean"),
    ).reset_index())

Completed NPZ files: 50
Partial NPZ files: 0

Status counts:


,status,videos
0,success,50



Per-class extraction summary:


,label,videos,pose_successes,sampled_frames,mean_pose_success_rate
0,BenchPress,10,228,267,0.887947
1,Biking,10,443,538,0.818803
2,BoxingPunchingBag,10,422,424,0.994970
3,HorseRiding,10,360,496,0.728672
4,TennisSwing,10,377,383,0.968129


---

# Stage B — Build fixed-length sequences for the LSTM

Run the next cells only after extraction has completed for the selected videos. This stage reads only complete per-video `.npz` files and excludes `.partial.npz` files.

**Sequence design:** 20 sampled poses per sequence, stride 5; 17 COCO joints × (x, y, confidence) = 51 features per frame. A window is kept when at least 80% of its frames have a valid detected-person pose. Missing frames are encoded as zeros after coordinate normalization.

**Leakage protection:** video-level train/validation/test splits are created **before** window generation, so windows from the same source video cannot cross splits. The selection step tries to use one clip from each UCF101 recording group per class, then fills from remaining clips only if necessary.

In [13]:

SEQUENCE_LENGTH = 16
SEQUENCE_STRIDE = 4
MIN_VALID_FRAME_RATIO = 0.80

TRAIN_FRACTION = 0.60
VAL_FRACTION = 0.20
TEST_FRACTION = 0.20
SPLIT_SEED = 42

assert abs(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION - 1.0) < 1e-8

complete_npz_files = sorted(
    p for p in BATCH_KEYPOINT_DIR.glob("*.npz")
    if ".partial." not in p.name and ".tmp." not in p.name
)
print("Complete per-video NPZ files found:", len(complete_npz_files))
assert complete_npz_files, "No complete per-video NPZ files. Run batch extraction first."

Complete per-video NPZ files found: 50


In [14]:
# Load video metadata. One entry here represents one source video, not one window.
video_records = []
for npz_path in complete_npz_files:
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            kps = d["keypoints"].astype(np.float32).copy()
            boxes = d["boxes"].astype(np.float32).copy()
            frame_indices = d["frame_indices"].astype(np.int32).copy()
            pose_ok = d["has_person"].astype(bool).copy()
            label = str(d["label"].item())
            video_path = str(d["video_path"].item())
        if kps.ndim != 3 or kps.shape[1:] != (17, 3):
            print("Skipping malformed keypoints:", npz_path.name, kps.shape)
            continue
        if len(kps) != len(boxes) or len(kps) != len(frame_indices) or len(kps) != len(pose_ok):
            print("Skipping inconsistent arrays:", npz_path.name)
            continue
        video_records.append({
            "npz_path": str(npz_path),
            "video_path": video_path,
            "label": label,
            "keypoints": kps,
            "boxes": boxes,
            "frame_indices": frame_indices,
            "pose_ok": pose_ok,
        })
    except Exception as exc:
        print("Skipping unreadable NPZ", npz_path.name, repr(exc))

print("Readable completed videos:", len(video_records))
print("Videos per class:")
display(pd.Series([r["label"] for r in video_records]).value_counts().rename_axis("label").reset_index(name="videos"))

Readable completed videos: 50
Videos per class:


,label,videos
0,BenchPress,10
1,Biking,10
2,BoxingPunchingBag,10
3,HorseRiding,10
4,TennisSwing,10


In [15]:
def normalize_video_poses(keypoints, boxes, pose_ok):
    """Normalize x/y within each frame's person box. Invalid frames become all zeros."""
    T = len(keypoints)
    normalized = np.zeros((T, 17, 3), dtype=np.float32)
    valid_frame = np.zeros(T, dtype=bool)

    for t in range(T):
        kp = keypoints[t]
        box = boxes[t]
        if not pose_ok[t]:
            continue
        if not np.isfinite(box).all() or not np.isfinite(kp[:, :2]).all():
            continue
        x1, y1, x2, y2 = map(float, box)
        bw, bh = x2 - x1, y2 - y1
        if bw < 2 or bh < 2:
            continue
        normalized[t, :, 0] = (kp[:, 0] - x1) / bw
        normalized[t, :, 1] = (kp[:, 1] - y1) / bh
        scores = kp[:, 2].copy()
        scores[~np.isfinite(scores)] = 0.0
        normalized[t, :, 2] = np.clip(scores, 0.0, 1.0)
        normalized[t] = np.nan_to_num(normalized[t], nan=0.0, posinf=0.0, neginf=0.0)
        valid_frame[t] = True

    return normalized, valid_frame

# Split videos by class before creating any windows.
rng = np.random.RandomState(SPLIT_SEED)
video_split = {}
video_split_rows = []
classes_present = sorted(set(r["label"] for r in video_records))

for label in classes_present:
    class_videos = sorted(
        [r["video_path"] for r in video_records if r["label"] == label]
    )
    if len(class_videos) < 5:
        raise ValueError(
            f"Only {len(class_videos)} completed videos for class {label}. "
            "Need at least 5 per class for a basic train/val/test split. "
            "Finish extraction or inspect the report first."
        )
    class_videos = list(np.asarray(class_videos, dtype=object)[rng.permutation(len(class_videos))])
    n = len(class_videos)
    n_test = max(1, int(round(n * TEST_FRACTION)))
    n_val = max(1, int(round(n * VAL_FRACTION)))
    if n_test + n_val >= n:
        n_test, n_val = 1, 1
    n_train = n - n_val - n_test
    assignments = (
        [(v, "train") for v in class_videos[:n_train]]
        + [(v, "val") for v in class_videos[n_train:n_train+n_val]]
        + [(v, "test") for v in class_videos[n_train+n_val:]]
    )
    for video_path, split in assignments:
        video_split[video_path] = split
        video_split_rows.append({"video_path": video_path, "label": label, "split": split})

video_split_df = pd.DataFrame(video_split_rows)
video_split_path = SEQUENCE_DIR / "video_split_manifest.csv"
video_split_df.to_csv(video_split_path, index=False)
print("Saved video split manifest:", video_split_path)
display(video_split_df.groupby(["label", "split"]).size().rename("videos").reset_index())

Saved video split manifest: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_5class\video_split_manifest.csv


,label,split,videos
0,BenchPress,test,2
1,BenchPress,train,6
2,BenchPress,val,2
3,Biking,test,2
4,Biking,train,6
5,Biking,val,2
6,BoxingPunchingBag,test,2
7,BoxingPunchingBag,train,6
8,BoxingPunchingBag,val,2
9,HorseRiding,test,2


In [16]:
# Build fixed-length windows after video-level split assignment.
X_by_split = {"train": [], "val": [], "test": []}
y_by_split = {"train": [], "val": [], "test": []}
video_by_split = {"train": [], "val": [], "test": []}
sequence_rows = []
sequence_count = 0

for rec in video_records:
    split = video_split[rec["video_path"]]
    normalized, valid_frame = normalize_video_poses(
        rec["keypoints"], rec["boxes"], rec["pose_ok"]
    )
    T = len(normalized)
    if T < SEQUENCE_LENGTH:
        print(f"Skipping {Path(rec['video_path']).name}: {T} samples < sequence length {SEQUENCE_LENGTH}")
        continue

    starts = list(range(0, T - SEQUENCE_LENGTH + 1, SEQUENCE_STRIDE))
    # Include the final possible window if the stride did not land on it.
    final_start = T - SEQUENCE_LENGTH
    if starts and starts[-1] != final_start:
        starts.append(final_start)

    for start in starts:
        end = start + SEQUENCE_LENGTH
        valid_ratio = float(valid_frame[start:end].mean())
        if valid_ratio < MIN_VALID_FRAME_RATIO:
            continue

        window = normalized[start:end].copy()
        # Invalid frames already have zero x/y/confidence values.
        window = np.nan_to_num(window, nan=0.0, posinf=0.0, neginf=0.0)
        X_by_split[split].append(window.reshape(SEQUENCE_LENGTH, 51))
        y_by_split[split].append(rec["label"])
        video_by_split[split].append(rec["video_path"])

        sequence_rows.append({
            "sequence_id": sequence_count,
            "video_path": rec["video_path"],
            "label": rec["label"],
            "split": split,
            "start_frame_index": int(rec["frame_indices"][start]),
            "end_frame_index": int(rec["frame_indices"][end - 1]),
            "valid_frame_ratio": round(valid_ratio, 4),
            "source_npz": rec["npz_path"],
        })
        sequence_count += 1

print("Sequence windows produced:", sequence_count)
for split in ["train", "val", "test"]:
    print(split, "windows:", len(X_by_split[split]),
          "unique videos:", len(set(video_by_split[split])))

if sequence_count == 0:
    raise ValueError(
        "No sequences were created. Check pose extraction success rates, "
        "and consider reducing SEQUENCE_LENGTH or MIN_VALID_FRAME_RATIO."
    )

Sequence windows produced: 307
train windows: 184 unique videos: 27
val windows: 73 unique videos: 10
test windows: 50 unique videos: 7


In [17]:
# Encode labels consistently, then save all arrays and metadata.
label_names = sorted(set(
    label for split in ["train", "val", "test"] for label in y_by_split[split]
))
label_to_id = {name: i for i, name in enumerate(label_names)}
label_map = {str(i): name for name, i in label_to_id.items()}

(SEQUENCE_DIR / "label_map.json").write_text(
    json.dumps(label_map, indent=2), encoding="utf-8"
)

for split in ["train", "val", "test"]:
    if X_by_split[split]:
        X_arr = np.asarray(X_by_split[split], dtype=np.float32)
        y_arr = np.asarray([label_to_id[label] for label in y_by_split[split]], dtype=np.int64)
        videos_arr = np.asarray(video_by_split[split], dtype=str)
    else:
        X_arr = np.empty((0, SEQUENCE_LENGTH, 51), dtype=np.float32)
        y_arr = np.empty((0,), dtype=np.int64)
        videos_arr = np.asarray([], dtype=str)

    np.save(SEQUENCE_DIR / f"X_{split}.npy", X_arr)
    np.save(SEQUENCE_DIR / f"y_{split}.npy", y_arr)
    np.save(SEQUENCE_DIR / f"videos_{split}.npy", videos_arr)
    print(f"Saved {split}: X={X_arr.shape}, y={y_arr.shape}, unique videos={len(set(videos_arr.tolist()))}")

sequence_manifest = pd.DataFrame(sequence_rows)
sequence_manifest_path = SEQUENCE_DIR / "sequence_manifest.csv"
sequence_manifest.to_csv(sequence_manifest_path, index=False)

# Explicit leakage check: no source video appears in more than one split.
sets = {s: set(video_by_split[s]) for s in ["train", "val", "test"]}
assert not (sets["train"] & sets["val"]), "Video leakage between train and validation!"
assert not (sets["train"] & sets["test"]), "Video leakage between train and test!"
assert not (sets["val"] & sets["test"]), "Video leakage between validation and test!"

print("\nSaved label map:", label_map)
print("Saved sequence manifest:", sequence_manifest_path)
print("Saved dataset directory:", SEQUENCE_DIR.resolve())
print("No source-video leakage detected between splits.")

Saved train: X=(184, 16, 51), y=(184,), unique videos=27
Saved val: X=(73, 16, 51), y=(73,), unique videos=10
Saved test: X=(50, 16, 51), y=(50,), unique videos=7

Saved label map: {'0': 'BenchPress', '1': 'Biking', '2': 'BoxingPunchingBag', '3': 'HorseRiding', '4': 'TennisSwing'}
Saved sequence manifest: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_5class\sequence_manifest.csv
Saved dataset directory: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_5class
No source-video leakage detected between splits.


In [18]:
# Final verification before opening 04_train_lstm.ipynb
print("DATASET SUMMARY")
print("Sequence length:", SEQUENCE_LENGTH)
print("Features per frame: 51 (17 joints x x/y/confidence)")
print("Classes:", label_map)

for split in ["train", "val", "test"]:
    X = np.load(SEQUENCE_DIR / f"X_{split}.npy")
    y = np.load(SEQUENCE_DIR / f"y_{split}.npy")
    vids = np.load(SEQUENCE_DIR / f"videos_{split}.npy")
    print(f"{split}: X={X.shape}, y={y.shape}, source videos={len(set(vids.tolist()))}")

print("\nVideos per class and split:")
display(video_split_df.groupby(["label", "split"]).size().rename("videos").reset_index())
print("\nSequences per class and split:")
if not sequence_manifest.empty:
    display(sequence_manifest.groupby(["label", "split"]).size().rename("sequences").reset_index())

print("\nNext: inspect the class counts, then open notebooks/04_train_lstm.ipynb.")
print("If any split has zero samples or a class is missing, do not train yet; send me this summary.")

DATASET SUMMARY
Sequence length: 16
Features per frame: 51 (17 joints x x/y/confidence)
Classes: {'0': 'BenchPress', '1': 'Biking', '2': 'BoxingPunchingBag', '3': 'HorseRiding', '4': 'TennisSwing'}
train: X=(184, 16, 51), y=(184,), source videos=27
val: X=(73, 16, 51), y=(73,), source videos=10
test: X=(50, 16, 51), y=(50,), source videos=7

Videos per class and split:


,label,split,videos
0,BenchPress,test,2
1,BenchPress,train,6
2,BenchPress,val,2
3,Biking,test,2
4,Biking,train,6
5,Biking,val,2
6,BoxingPunchingBag,test,2
7,BoxingPunchingBag,train,6
8,BoxingPunchingBag,val,2
9,HorseRiding,test,2



Sequences per class and split:


,label,split,sequences
0,BenchPress,test,2
1,BenchPress,train,21
2,BenchPress,val,7
3,Biking,test,5
4,Biking,train,57
5,Biking,val,16
6,BoxingPunchingBag,test,14
7,BoxingPunchingBag,train,47
8,BoxingPunchingBag,val,20
9,HorseRiding,test,10



Next: inspect the class counts, then open notebooks/04_train_lstm.ipynb.
If any split has zero samples or a class is missing, do not train yet; send me this summary.


## Outputs created by this notebook

- `data/processed/selected_5class_videos.csv` — the selected source videos.
- `data/processed/keypoints_batch_5class/*.npz` — one pose file per successfully extracted video; partial `.npz` files are retained when interrupted.
- `data/processed/batch_5class_extraction_report.csv` — detection and pose success counts per video.
- `outputs/pose_videos/batch_5class_previews/` — one skeleton preview per video when a valid pose is found.
- `data/processed/sequences_5class/X_train.npy`, `X_val.npy`, `X_test.npy` — LSTM inputs shaped `(N, 20, 51)`.
- Corresponding `y_*.npy`, `videos_*.npy`, `label_map.json`, `video_split_manifest.csv`, and `sequence_manifest.csv`.

**Important limitation:** this is an initial five-class baseline. It selects the largest detection independently per frame and does not guarantee person identity through time. It also uses the folder/video action label for each sampled window, which assumes the labelled action dominates that video. The evaluation set is split by source video to prevent overlapping windows from leaking across splits. For strong results, inspect low-quality samples and expand beyond 10 videos per class after this pipeline is stable.